# 01_Silver_Customers

## Objective

This notebook transforms the Bronze Customers dataset into a clean,
standardized, and business-ready Silver Customers dataset.

The Silver layer focuses on improving data quality by:
- Standardizing text fields
- Cleaning inconsistent values
- Applying business validations
- Removing duplicate records
- Preparing the dataset for downstream analytics

The final Silver table serves as the trusted customer dimension
for the Gold layer.

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# %run ../00_Setup/00_Environment_Initialization

In [0]:
from delta.tables import DeltaTable

In [0]:
# Metadata

PIPELINE_NAME = "Silver_Customers_Load"
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

Pipeline Started : 2026-07-19 17:55:50.246260


In [0]:
# Configuration

SOURCE_TABLE = TARGET_TABLE
TARGET_TABLE = SILVER_CUSTOMER

In [0]:
# Read Bronze Customers

customers_df = spark.table(SOURCE_TABLE)

In [0]:
total_rows = customers_df.count()

print("DATASET PROFILE")
print(f"Rows    : {total_rows}")
print(f"Columns : {len(customers_df.columns)}")
customers_df.printSchema()
display(customers_df.limit(10))

DATASET PROFILE
Rows    : 15000
Columns : 12
root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- customer_lifetime_days: integer (nullable = true)
 |-- days_since_last_purchase: integer (nullable = true)
 |-- churn_risk_segment: string (nullable = true)



customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days,days_since_last_purchase,churn_risk_segment
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809,0,Active
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415,425,High Risk - Likely Churned
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370,371,High Risk - Likely Churned
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741,60,Active
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0,316,High Risk - Likely Churned
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559,325,High Risk - Likely Churned
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0,null,Never Purchased
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849,20,Active
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0,742,High Risk - Likely Churned
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222,364,High Risk - Likely Churned


# Data Cleaning and Standardization 

In [0]:
# Data Cleaning - Trim Whitespace

from pyspark.sql import functions as F

customers_df = (
    customers_df
    .withColumn(
        "customer_city",
        F.trim(F.col("customer_city"))
    )
    .withColumn(
        "customer_state",
        F.trim(F.col("customer_state"))
    )
)

In [0]:
# Data Standardization

customers_df = (
    customers_df
    .withColumn(
        "customer_city",
        F.initcap(F.col("customer_city"))
    )
    .withColumn(
        "customer_state",
        F.upper(F.col("customer_state"))
    )
)

# Validation check — confirm no remaining whitespace
remaining_spaces = customers_df.filter(
    (F.col("customer_city") != F.trim(F.col("customer_city"))) |
    (F.col("customer_state") != F.trim(F.col("customer_state")))
).count()

standardization_status = (
    "PASSED"
    if remaining_spaces == 0
    else "FAILED"
)

In [0]:
# Remove Duplicate Records

before_count = customers_df.count()
customers_df = customers_df.dropDuplicates(["customer_id"])
after_count = customers_df.count()
duplicates_removed = before_count - after_count
print(f"Duplicates Removed : {duplicates_removed}")

Duplicates Removed : 0


In [0]:
duplicate_status = (
    "PASSED"
    if duplicates_removed == 0
    else f"FAILED ({duplicates_removed} duplicates removed)"
)

In [0]:
display(
    customers_df.select(
        "customer_id",
        "customer_city",
        "customer_state"
    ).limit(10)
)

customer_id,customer_city,customer_state
CUST_000018,Manaus,AM
CUST_000036,Manaus,AM
CUST_000066,Campo Grande,MS
CUST_000089,Joao Pessoa,PB
CUST_000095,Teresina,PI
CUST_000105,Recife,PE
CUST_000117,Manaus,AM
CUST_000119,Teresina,PI
CUST_000136,Joao Pessoa,PB
CUST_000159,Porto Velho,RO


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailmart.quarantine;

# Bussiness Validation 

In [0]:
# valid customers 
valid_customers = customers_df.filter(
    F.col("customer_id").isNotNull() &
    F.col("customer_city").isNotNull() &
    F.col("customer_state").isNotNull()
)

In [0]:
# invalid customers
invalid_customers = customers_df.filter(
    F.col("customer_id").isNull() |
    F.col("customer_city").isNull() |
    F.col("customer_state").isNull()
)

In [0]:
invalid_customers = (
    invalid_customers
    .withColumn(
        "quarantine_reason",
        F.when(F.col("customer_id").isNull(), "Missing Customer ID")
         .when(F.col("customer_city").isNull(), "Missing Customer City")
         .when(F.col("customer_state").isNull(), "Missing Customer State")
    )
)

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.QURANTINE_CUSTOMERS;

In [0]:
# Qurantine table
(
    invalid_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_CUSTOMERS)
)

In [0]:
customers_df = valid_customers

In [0]:
# Create Temporary View

customers_df.createOrReplaceTempView("customers_updates")

In [0]:
valid_customers = add_audit_columns(valid_customers,PIPELINE_NAME,RUN_ID)

In [0]:
# Before MERGE, check whether the table exists.
table_exists = spark.catalog.tableExists(TARGET_TABLE)

Because our dataset is static.
we use overrite instead of append.
Every time we rerun the notebook during development, we'll otherwise accumulate the same invalid records repeatedly.

In [0]:
#Databricks SQL doesn't substitute Python variables inside %sql.
#Instead, we'll use Python

# Write to silver layer - (USING MERGE)


operation = ""
try:

    if not table_exists:
        (
            valid_customers.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(TARGET_TABLE)
        )
        operation = "CREATED"
    else:
        delta_table = DeltaTable.forName(
            spark,
            TARGET_TABLE
        )
        (
            delta_table.alias("target")
            .merge(
                valid_customers.alias("source"),
                "target.customer_id = source.customer_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        operation = "MERGED"
except Exception as err:
    operation = "FAILED"
    raise

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7717810061639207>, line 31
     18     else:
     19         delta_table = DeltaTable.forName(
     20             spark,
     21             TARGET_TABLE
     22         )
     23         (
     24             delta_table.alias("target")
     25             .merge(
     26                 valid_customers.alias("source"),
     27                 "target.customer_id = source.customer_id"
     28             )
     29             .whenMatchedUpdateAll()
     30             .whenNotMatchedInsertAll()
---> 31             .execute()
     32         )
     33         operation = "MERGED"
     34 except Exception as err:

File /databricks/python/lib/python3.12/site-packages/delta/connect/tables.py:609, in DeltaMergeBuilder.execute(self)
    599 plan = MergeIntoTable(
    600     self._target,
    601     self._source,
   (...)
  

In [0]:
Aexecution_status = (
    "SUCCESS"
    if operation != "FAILED"
    else "FAILED"
)

In [0]:
silver_df = spark.table(TARGET_TABLE)
rows_written = silver_df.count()
print(f"Rows Written : {rows_written}")
display(silver_df)

Rows Written : 15000


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,ingestion_date,pipeline_name,run_id
CUST_000018,UNIQ_000018,54118,Manaus,AM,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000036,UNIQ_000036,97841,Manaus,AM,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000066,UNIQ_000066,79352,Campo Grande,MS,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000089,UNIQ_000089,19071,Joao Pessoa,PB,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000095,UNIQ_000095,37760,Teresina,PI,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000105,UNIQ_000105,10942,Recife,PE,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000117,UNIQ_000117,22363,Manaus,AM,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000119,UNIQ_000119,56438,Teresina,PI,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000136,UNIQ_000136,73092,Joao Pessoa,PB,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74
CUST_000159,UNIQ_000159,20745,Porto Velho,RO,2026-07-16T09:25:40.511Z,2026-07-16,Silver_Customers_Load,695602f5-0758-4487-9534-d4e59b319e74


In [0]:
valid_records = valid_customers.count()
invalid_records = invalid_customers.count()
data_quality_status = (
    "PASSED"
    if invalid_records == 0
    else f"FAILED ({invalid_records} quarantined)"
)

In [0]:
print("VALIDATION SUMMARY")
print(f"Rows Read              : {total_rows}")
print(f"Valid Records          : {valid_records}")
print(f"Quarantined Records    : {invalid_records}")
print(f"Rows Written           : {total_rows}")
print(f"Duplicate Removal      : {duplicate_status}")

VALIDATION SUMMARY
Rows Read              : 15000
Valid Records          : 15000
Quarantined Records    : 0
Rows Written           : 15000
Duplicate Removal      : PASSED


In [0]:
# Silver Load Report

END_TIME = end_pipeline()
duration = execution_time(START_TIME,END_TIME)
print("SILVER LOAD REPORT")
print(f"Pipeline Name        : {PIPELINE_NAME}")
print(f"Run ID               : {RUN_ID}")
print(f"Source Table         : {SOURCE_TABLE}")
print(f"Target Table         : {TARGET_TABLE}")
print(f"Rows Read            : {total_rows}")
print(f"Valid Records        : {valid_records}")
print(f"Quarantined Records  : {invalid_records}")
print(f"Rows Written         : {total_rows}")
print(f"Start Time           : {START_TIME}")
print(f"End Time             : {END_TIME}")
print(f"Duration             : {duration} sec")
print(f"Load Status          : {execution_status}")
print(f"Operation            : {operation}")
print(f"Standardization Check: {standardization_status}")

Pipeline Completed : 2026-07-19 18:04:00.826922
SILVER LOAD REPORT
Pipeline Name        : Silver_Customers_Load
Run ID               : 2ae93554-4c1a-499b-987d-b5573a64b280
Source Table         : retailmart.bronze.customers
Target Table         : retailmart.silver.customers
Rows Read            : 15000
Valid Records        : 15000
Quarantined Records  : 0
Rows Written         : 15000
Start Time           : 2026-07-19 17:55:50.246260
End Time             : 2026-07-19 18:04:00.826922
Duration             : 490.58 sec
Load Status          : FAILED
Operation            : FAILED
Standardization Check: PASSED


# Engineering Observations

- Customer city values were trimmed and standardized using `trim()` and `initcap()`.
- Customer state values were converted to uppercase for consistency.
- Duplicate customer records were removed using `customer_id` as the business key.
- Invalid customer records were redirected to the Quarantine layer with a descriptive `quarantine_reason`.
- Silver audit columns were refreshed to reflect the Silver pipeline execution.
- Delta Lake MERGE was used to support incremental SCD Type 1 loading.
- The resulting Silver table is standardized, validated, and ready for downstream consumption.